In [1]:
from pathlib import Path
import json
from time import perf_counter
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor

BASE = Path("experiment_v2") if Path("experiment_v2/data").is_dir() else Path(".")

model_data = xr.open_dataset(BASE / "data/processed/malta_forecast_prepared.nc")

feature_names = json.loads(model_data.attrs["feature_names_json"])
horizons = tuple(int(h) for h in model_data.attrs["forecast_horizons_days"])
model_mask = model_data["model_mask"].values == 1

# Share of training rows used for fitting. Neighbouring pixels on the same
# day are strongly correlated, so a random subset keeps every day while
# cutting memory and fit time. Set to 1.0 to use every row.
TRAIN_SAMPLE_FRACTION = 0.6

print(f"Predictors available: {len(feature_names)}")
print(f"Forecast horizons: {horizons}")
print(f"Training period: {model_data.attrs['training_period']}")


Predictors available: 21
Forecast horizons: (1, 4, 7)
Training period: 2017–2023


In [2]:
def make_samples(splits, horizon, predictors=None, fraction=1.0, seed=0):
    """Rows = (issue day, pixel) with an observed target `horizon` days later."""
    predictors = feature_names if predictors is None else predictors
    splits = [splits] if isinstance(splits, str) else splits

    in_split = np.zeros(model_data.sizes["time"], dtype=bool)
    for split in splits:
        in_split |= model_data[f"split_{split}"].values == 1
    days = np.flatnonzero(in_split)

    # Read one contiguous block of dates: netCDF reads a list of dates with
    # gaps one day at a time, which is hundreds of times slower. Days in the
    # block that belong to no requested split are excluded through `valid`.
    block = slice(days[0], days[-1] + 1)
    subset = model_data.isel(time=block)
    in_block = in_split[block]

    def grid(name, data=subset):
        return data[name].transpose("time", "latitude", "longitude").values

    target = grid(f"chl_target_{horizon}d")
    valid = (
        np.isfinite(target) & (target > 0)
        & model_mask[None] & in_block[:, None, None]
    )
    t, i, j = np.nonzero(valid)

    if fraction < 1.0:
        keep = np.random.default_rng(seed).random(t.size) < fraction
        t, i, j = t[keep], i[keep], j[keep]

    columns = {}
    for name in predictors:
        dims = subset[name].dims
        if dims == ("time",):
            columns[name] = subset[name].values[t]
        elif dims == ("latitude",):
            columns[name] = subset[name].values[i]
        elif dims == ("longitude",):
            columns[name] = subset[name].values[j]
        else:
            columns[name] = grid(name)[t, i, j]

    # Climatology for the target date: the issue-month value h days later.
    target_clim = model_data["chl_clim_log"].shift(time=-horizon).isel(time=block)

    return {
        "X": pd.DataFrame(columns).astype("float32"),
        "observed": target[t, i, j].astype("float64"),
        "persistence": grid("chl_latest")[t, i, j].astype("float64"),
        "clim_log": grid("chl_clim_log", target_clim.to_dataset())[t, i, j]
            .astype("float64"),
        "day": t,
        "target_month": (
            pd.DatetimeIndex(subset.time.values[t])
            + pd.Timedelta(days=horizon)
        ).month.to_numpy(),
    }

started = perf_counter()
check = make_samples("validation", 1)
print(f"Validation +1d rows: {len(check['observed']):,} "
      f"({perf_counter() - started:.0f} s)")
display(check["X"].describe().T[["mean", "min", "max"]].round(3))


Validation +1d rows: 2,506,146 (4 s)


,mean,min,max
season_sin,-0.054,-1.000,1.000
season_cos,-0.081,-1.000,1.000
chl_latest,0.067,0.006,3.137
chl_age_days,1.856,1.000,74.000
sst_c,22.769,14.090,30.410
sst_observed,0.669,0.000,1.000
wind_u_asc,2.115,-16.540,17.240
wind_v_asc,-1.495,-13.880,13.960
wind_speed_asc,6.044,0.240,18.110
wind_asc_observed,0.378,0.000,1.000


In [3]:
def chl_scores(observed, predicted):
    observed = np.asarray(observed, dtype="float64")
    predicted = np.asarray(predicted, dtype="float64")
    error = predicted - observed
    log_error = np.log10(predicted) - np.log10(observed)
    return {
        "Scored pixel-days": observed.size,
        "Bias (mg/m³)": error.mean(),
        "MAE (mg/m³)": np.abs(error).mean(),
        "RMSE (mg/m³)": np.sqrt(np.mean(error ** 2)),
        "RMSE (log10)": np.sqrt(np.mean(log_error ** 2)),
        "Median relative error (%)": np.median(100 * np.abs(error) / observed),
    }


def log_rmse_gain_interval(observed, model, baseline, day, n_boot=1000, seed=0):
    """% reduction in log10 RMSE versus a baseline, with a 95% interval
    from resampling whole issue days (pixels on one day are not independent)."""
    log_observed = np.log10(observed)
    _, day_id = np.unique(day, return_inverse=True)
    n_days = day_id.max() + 1

    def daily_sums(predicted):
        squared = (np.log10(predicted) - log_observed) ** 2
        return np.bincount(day_id, squared, n_days)

    model_sum, baseline_sum = daily_sums(model), daily_sums(baseline)
    counts = np.bincount(day_id, minlength=n_days)

    def gain(weights):
        n = weights @ counts
        return 100 * (1 - np.sqrt(weights @ model_sum / n)
                      / np.sqrt(weights @ baseline_sum / n))

    rng = np.random.default_rng(seed)
    resampled = [
        gain(np.bincount(rng.integers(0, n_days, n_days), minlength=n_days))
        for _ in range(n_boot)
    ]
    low, high = np.percentile(resampled, [2.5, 97.5])
    return gain(np.ones(n_days)), low, high


def fit_damped_persistence(sample):
    """log10(future) = a + b·log10(latest) + c·climatology, fitted on training rows."""
    design = np.column_stack([
        np.ones(sample["observed"].size),
        np.log10(sample["persistence"]),
        sample["clim_log"],
    ])
    usable = np.isfinite(design).all(axis=1)
    coefficients, *_ = np.linalg.lstsq(
        design[usable], np.log10(sample["observed"][usable]), rcond=None
    )
    return coefficients


def baseline_predictions(sample, damped_coefficients):
    latest_log = np.log10(sample["persistence"])
    damped_log = (
        damped_coefficients[0]
        + damped_coefficients[1] * latest_log
        + damped_coefficients[2] * sample["clim_log"]
    )
    # Pixels without CHL history fall back to climatology.
    damped_log = np.where(np.isfinite(latest_log), damped_log, sample["clim_log"])
    return {
        "Persistence": sample["persistence"],
        "Climatology (2017–2023)": 10.0 ** sample["clim_log"],
        "Damped persistence": 10.0 ** damped_log,
    }


In [4]:
CORE_FEATURES = [
    "season_sin", "season_cos",
    "chl_latest", "chl_age_days",
    "chl_mean_7d", "chl_observations_7d",
    "latitude_deg", "longitude_deg",
]

FEATURE_SETS = {
    "Core: CHL history + season + location": CORE_FEATURES,
    "Core + climatology": CORE_FEATURES + ["chl_clim_log"],
    "Core + climatology + SST":
        CORE_FEATURES + ["chl_clim_log", "sst_c", "sst_observed"],
    f"All {len(feature_names)} features": feature_names,
}

MODEL_PARAMS = dict(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=100,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42,
)

validation_rows = []
damped_coefficients = {}

for horizon in horizons:
    started = perf_counter()
    train = make_samples("train", horizon, fraction=TRAIN_SAMPLE_FRACTION)
    valid = make_samples("validation", horizon)

    damped_coefficients[horizon] = fit_damped_persistence(train)
    predictions = baseline_predictions(valid, damped_coefficients[horizon])

    for name, predictors in FEATURE_SETS.items():
        model = HistGradientBoostingRegressor(**MODEL_PARAMS)
        model.fit(train["X"][predictors], np.log10(train["observed"]))
        predictions[name] = 10.0 ** model.predict(valid["X"][predictors])

    for method, predicted in predictions.items():
        validation_rows.append({
            "Horizon": f"+{horizon}d",
            "Method": method,
            **chl_scores(valid["observed"], predicted),
        })

    print(f"+{horizon}d: {len(train['observed']):,} training rows, "
          f"coefficients {np.round(damped_coefficients[horizon], 3)}, "
          f"{(perf_counter() - started) / 60:.1f} min")
    del train

validation_results = pd.DataFrame(validation_rows).set_index(["Horizon", "Method"])


+1d: 10,845,438 training rows, coefficients [0.027 0.494 0.525], 2.1 min
+4d: 10,840,808 training rows, coefficients [0.039 0.34  0.687], 2.1 min
+7d: 10,844,827 training rows, coefficients [0.042 0.26  0.769], 2.3 min


In [5]:
baseline_names = ["Persistence", "Climatology (2017–2023)", "Damped persistence"]

best_baseline = (
    validation_results["RMSE (log10)"]
    .unstack("Method")[baseline_names]
    .min(axis=1)
)
validation_results["Log RMSE gain vs best baseline (%)"] = 100 * (
    1 - validation_results["RMSE (log10)"]
    / best_baseline.reindex(validation_results.index, level="Horizon")
)

display(validation_results.style.format({
    "Scored pixel-days": "{:,.0f}",
    "Bias (mg/m³)": "{:+.4f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
    "Log RMSE gain vs best baseline (%)": "{:+.1f}",
}))


In [6]:
FINAL_SET = "Core + climatology + SST"
FINAL_FEATURES = FEATURE_SETS[FINAL_SET]
FINAL_SPLITS = ["train", "validation"]   # issue dates 2008-01-01 to 2024-12-24

final_models = {}
final_damped = {}
test_samples = {}
test_predictions = {}

for horizon in horizons:
    started = perf_counter()
    fit_rows = make_samples(FINAL_SPLITS, horizon, predictors=FINAL_FEATURES, fraction=TRAIN_SAMPLE_FRACTION)

    final_damped[horizon] = fit_damped_persistence(fit_rows)
    final_models[horizon] = HistGradientBoostingRegressor(**MODEL_PARAMS)
    final_models[horizon].fit(
        fit_rows["X"][FINAL_FEATURES], np.log10(fit_rows["observed"])
    )
    del fit_rows

    test = make_samples("test", horizon, predictors=FINAL_FEATURES)
    predictions = baseline_predictions(test, final_damped[horizon])
    predictions["Forecast model"] = 10.0 ** final_models[horizon].predict(
        test["X"][FINAL_FEATURES]
    )

    test_samples[horizon] = test
    test_predictions[horizon] = predictions
    print(f"+{horizon}d: fitted and scored on {len(test['observed']):,} test "
          f"pixel-days in {(perf_counter() - started) / 60:.1f} min")

test_results = pd.DataFrame([
    {"Horizon": f"+{h}d", "Method": method,
     **chl_scores(test_samples[h]["observed"], predicted)}
    for h in horizons
    for method, predicted in test_predictions[h].items()
]).set_index(["Horizon", "Method"])

score_formats = {
    "Scored pixel-days": "{:,.0f}",
    "Bias (mg/m³)": "{:+.4f}",
    "MAE (mg/m³)": "{:.4f}",
    "RMSE (mg/m³)": "{:.4f}",
    "RMSE (log10)": "{:.4f}",
    "Median relative error (%)": "{:.1f}",
}
display(test_results.style.format(score_formats))


+1d: fitted and scored on 2,835,920 test pixel-days in 0.7 min
+4d: fitted and scored on 2,845,584 test pixel-days in 0.7 min
+7d: fitted and scored on 2,850,791 test pixel-days in 0.7 min


In [7]:
interval_rows = []

for horizon in horizons:
    test = test_samples[horizon]
    model = test_predictions[horizon]["Forecast model"]

    for baseline in baseline_names:
        gain, low, high = log_rmse_gain_interval(
            test["observed"], model,
            test_predictions[horizon][baseline],
            test["day"],
        )
        interval_rows.append({
            "Horizon": f"+{horizon}d",
            "Compared with": baseline,
            "Log RMSE gain (%)": gain,
            "95% interval low": low,
            "95% interval high": high,
            "Clearly better": low > 0,
        })

gain_intervals = pd.DataFrame(interval_rows).set_index(["Horizon", "Compared with"])
display(gain_intervals.style.format({
    "Log RMSE gain (%)": "{:+.1f}",
    "95% interval low": "{:+.1f}",
    "95% interval high": "{:+.1f}",
}))


In [8]:
season_of_month = {12: "Winter", 1: "Winter", 2: "Winter",
                   3: "Spring", 4: "Spring", 5: "Spring",
                   6: "Summer", 7: "Summer", 8: "Summer",
                   9: "Autumn", 10: "Autumn", 11: "Autumn"}

# "High CHL": observed at least 1.6× (0.2 in log10) the target-month climatology.
HIGH_ANOMALY_LOG10 = 0.2

rows = []
for horizon in horizons:
    test = test_samples[horizon]
    seasons = pd.Series(test["target_month"]).map(season_of_month).to_numpy()
    high = np.log10(test["observed"]) - test["clim_log"] >= HIGH_ANOMALY_LOG10

    groups = {name: seasons == name for name in ["Winter", "Spring", "Summer", "Autumn"]}
    groups["High CHL (≥1.6× normal)"] = high

    for group, selected in groups.items():
        for method in ["Persistence", "Damped persistence", "Forecast model"]:
            rows.append({
                "Horizon": f"+{horizon}d",
                "Subset": group,
                "Method": method,
                **chl_scores(test["observed"][selected],
                             test_predictions[horizon][method][selected]),
            })

subset_results = pd.DataFrame(rows).set_index(["Horizon", "Subset", "Method"])
display(subset_results.style.format(score_formats))


In [9]:
import joblib
import sklearn
from datetime import datetime, timezone

model_dir = BASE / "models"
model_dir.mkdir(parents=True, exist_ok=True)
bundle_path = model_dir / "chl_forecast_1_4_7d.joblib"

# One climatology map per calendar month, for baselines and for pixels
# without CHL history in operational use.
issue_dates = pd.DatetimeIndex(model_data.time.values)
first_day_of_month = [np.flatnonzero(issue_dates.month == m)[0] for m in range(1, 13)]
monthly_log_climatology = (
    model_data["chl_clim_log"]
    .isel(time=first_day_of_month)
    .assign_coords(month=("time", np.arange(1, 13)))
    .swap_dims(time="month")
    .drop_vars("time")
    .load()
)

fit_days = np.flatnonzero(
    (model_data["split_train"].values == 1) | (model_data["split_validation"].values == 1)
)

model_bundle = {
    "models": final_models,
    "feature_names": list(FINAL_FEATURES),
    "feature_set": FINAL_SET,
    "forecast_horizons_days": list(horizons),
    "forecast_issue_time_utc": model_data.attrs["forecast_issue_time_utc"],
    "target_transform": "log10",
    "target_units": "milligram m^-3",
    "preprocessing": {
        "chl_availability_lag_days": 1,
        "sst_availability_lag_days": 0,
        "chl_history_window_days": 7,
        "sst_units": "degree_Celsius",
        "max_plausible_chl_mg_m3": 10.0,
        "min_training_observations_per_pixel": 200,
    },
    "training_issue_date_range": [
        str(issue_dates[fit_days[0]].date()),
        str(issue_dates[fit_days[-1]].date()),
    ],
    "train_sample_fraction": TRAIN_SAMPLE_FRACTION,
    "availability_basis": model_data.attrs["availability_basis"],
    "model_mask": model_data["model_mask"].load().copy(deep=True),
    "monthly_log_climatology": monthly_log_climatology,
    "climatology_period": "2017–2023",
    "damped_persistence_coefficients": {h: c.tolist() for h, c in final_damped.items()},
    "test_year": 2025,
    "validation_results": validation_results.copy(),
    "test_results": test_results.copy(),
    "test_gain_intervals": gain_intervals.copy(),
    "sklearn_version": sklearn.__version__,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

joblib.dump(model_bundle, bundle_path, compress=3)

reloaded = joblib.load(bundle_path)
sample_inputs = test_samples[1]["X"][FINAL_FEATURES].iloc[:100]
for horizon in horizons:
    np.testing.assert_allclose(
        reloaded["models"][horizon].predict(sample_inputs),
        final_models[horizon].predict(sample_inputs),
        rtol=1e-12, atol=1e-12,
    )

print(f"Saved and verified: {bundle_path} "
      f"({bundle_path.stat().st_size / 1024**2:,.2f} MB)")


Saved and verified: models/chl_forecast_1_4_7d.joblib (1.18 MB)


## Training-era experiment (2024 validation, log10 RMSE)

Trained on 2008–2023 versus 2017–2023 (Sentinel-3 era), 30% row sample, same 2024 validation rows.

| Method | Trained on | +1d | +4d | +7d |
|---|---|---|---|---|
| Climatology | 2008–2023 | 0.1309 | 0.1312 | 0.1319 |
| Climatology | 2017–2023 | 0.1240 | 0.1245 | 0.1255 |
| Damped persistence | 2008–2023 | 0.1105 | 0.1232 | 0.1306 |
| Damped persistence | 2017–2023 | 0.1064 | 0.1178 | 0.1245 |
| Core + SST | 2008–2023 | 0.1018 | 0.1171 | 0.1260 |
| Core + SST | 2017–2023 | 0.1021 | 0.1155 | 0.1242 |
| Core + climatology + SST | 2008–2023 | 0.1020 | 0.1183 | 0.1274 |
| Core + climatology + SST | 2017–2023 | 0.1009 | 0.1153 | 0.1243 |

The 2017–2023 climatology has half the bias (+0.004 vs +0.008 mg/m³): CHL in recent years is lower than in 2008–2016.
With half the rows, recent-only training matches or beats 2008–2023 for every method, so the final model uses 2017 onwards.
Against recent damped persistence the model gains 5.2% (+1d), 2.1% (+4d) and 0.2% (+7d): little skill beyond a persistence–climatology blend at +7 days.
